# Artificial Intelligence
## L.EIC – 3rd Year/2nd Semester
### Exercise Sheet 1
# Solving Problems by Searching

## The Two Buckets Problem

<img src="https://qph.cf2.quoracdn.net/main-qimg-45726b16b460cae0147ae8ca245a8fb0-pjlq" width="250px" height="250px" align="right">

Two buckets of capacities **c1** (e.g. 4 liters) and **c2** (e.g. 3 liters), respectively, are initially empty. The buckets do not have any intermediate markings. The only operations you can perform are:

- Fill (completely) a bucket
- Empty a bucket.
- Pour one bucket into the other until the target one is full or the source is empty.

The aim is to determine which operations to carry out so that the first bucket contains exactly **n** liters (e.g. 2 litres).

**a)** Formulate this problem as a search problem by defining the state representation, initial state, operators (their name, preconditions, effects, and cost), and final state(s).

**b)** Draw a partial state space graph with some of the possible transitions. What is the state space size for this problem? Consider only reachable states. *(Note: you might want to do this on paper and pencil.)*

c1*c2

**c)** Solve the problem by hand, using a search tree. What solutions have you found?

hell nah

### Buildind a computational approach to handle the problem

*(Note: this notebook suggests an implementation using Python; however, feel free to use any other programming language you like, such as Prolog ;-) .)*

To build a program to solve the buckets problem, we will implement a solution that separates the problem definition from the algorithms used to traverse the state space. This way, we can reuse our implementations of the search strategies in other problems.

#### Representing the two buckets problem as a search problem

Let's start by defining a state for the buckets problem. For that, it'll suffice to aggregate two quantities, each representing the amount of water in one of the buckets. We also define a way of printing the state.

In [21]:
class BucketState:
    c1 = 4   # capacity for bucket 1
    c2 = 3   # capacity for bucket 2

    def __init__(self, b1, b2):
        self.b1 = b1
        self.b2 = b2

    '''needed for the visited list'''
    def __eq__(self, other):
        if isinstance(other, self.__class__):
            return self.__dict__ == other.__dict__
        else:
            return False

    def __ne__(self, other):
        """Overrides the default implementation (unnecessary in Python 3)"""
        return not self.__eq__(other)

    def __hash__(self):
        return hash((self.b1, self.b2))
    ''' - '''

    def __str__(self):
        return "(" + str(self.b1) + ", " + str(self.b2) + ")"

Now we define each of the operators on states:

In [22]:
# emptying the first bucket
def empty1(state):
    if state.b1 > 0:
        return BucketState(0, state.b2), 1   # using unitary costs
    return None

# emptying the second bucket
def empty2(state):
    if state.b2 > 0:
        return BucketState(state.b1, 0), 1   # using unitary costs
    return None




# filling the first bucket
def fill1(state):
    if state.b1 < BucketState.c1:
        return BucketState(BucketState.c1, state.b2), 1
    return None

# filling the second bucket
def fill2(state):
    if state.b2 < BucketState.c2:
        return BucketState(state.b1, BucketState.c2), 1
    return None




# pouring from the first to the second bucket until the target is full
def pour12_fill2(state):
    if state.b1 > 0 and state.b2 < BucketState.c2 and state.b1+state.b2 >= BucketState.c2:
        return BucketState(state.b1 - (BucketState.c2 - state.b2), BucketState.c2), 1
    return None

# pouring from the first to the second bucket until the source is empty
def pour12_empty1(state):
    if state.b1 > 0 and state.b2 < BucketState.c2 and state.b1+state.b2 < BucketState.c2:
        return BucketState(0, state.b1 + state.b2), 1
    return None




# pouring from the second to the first bucket until the target is full
def pour21_fill1(state):
    if state.b2 > 0 and state.b1< BucketState.c1 and state.b2+state.b1 >= BucketState.c1:
        return BucketState(BucketState.c1, state.b2 - (BucketState.c1 - state.b1)), 1
    return None




# pouring from the second to the first bucket until the source is empty
def pour21_empty2(state):
    if state.b2 > 0 and state.b1 < BucketState.c2 and state.b1+state.b2 < BucketState.c1:
        return BucketState(state.b1 + state.b2, 0), 1
    return None




The following function will aggregate all states that can be generated from a given one:

In [ ]:
def child_bucket_states(state):
    new_states = []
    if(empty1(state)):
        new_states.append(empty1(state))
    if(empty2(state)):
        new_states.append(empty2(state))
    if(fill1(state)):
        new_states.append(fill1(state))
    if(fill2(state)):
        new_states.append(fill2(state))
    if(pour12_fill2(state)):
        new_states.append(pour12_fill2(state))
    if(pour12_empty1(state)):
        new_states.append(pour12_empty1(state))
    if(pour21_fill1(state)):
        new_states.append(pour21_fill1(state))
    if(pour21_empty2(state)):
        new_states.append(pour21_empty2(state))
    return new_states

Play around with the state transition operators and check if they are working properly:

In [24]:
s = BucketState(0, 0)
s,_ = fill1(s)
s,_ = fill2(s)
#print(s)

s,_ = empty1(s)
s,_ = empty2(s)

#print(s)

s,_ = fill1(s)
s,_ = pour12_fill2(s)

#print(s)

s,_ = empty1(s)
s,_ = empty2(s)

s,_ = fill2(s)
s,_ = pour21_empty2(s)

#print(s)

s,_ = fill2(s)
s,_ = pour21_fill1(s)

s,_ = empty1(s)
s,_ = pour21_empty2(s)

#print(s)

s,_ = pour12_empty1(s)

#print(s)
for child,_ in child_bucket_states(BucketState(2, 0)): 
    print(child)


(0, 0)
(4, 0)
(2, 3)
(0, 2)


Finally, we need to define the goal condition:

In [25]:
def goal_bucket_state(state):
    if (state.b1 == 2):
        return state
    return None




Test your goal condition:

In [26]:
s = BucketState(1,5)
print(goal_bucket_state(s))



None


#### Implementing search algorithms

Let us start by defining an appropriate structure to represent a node in a search tree. Each tree node will include:
- a state of the problem
- a link to its parent (to allow traveling from a leaf node towards the root of the tree)
- a list of child nodes

In [27]:
# A generic definition of a tree node holding a state of the problem
class TreeNode:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = []

    def add_child(self, child_node):
        self.children.append(child_node)
        child_node.parent = self

##### Breadth-first search

Based on this structure, we can now implement breadth-first search. Note that we want the implementation to be independent of the problem at hand (in this case, the two buckets problem).

In [28]:
from collections import deque

def breadth_first_search(initial_state, goal_state_func, operators_func):
    root = TreeNode(initial_state)   # create the root node in the search tree
    queue = deque([root])   # initialize the queue to store the nodes

    while queue:
        node = queue.popleft()   # get first element in the queue
        if goal_state_func(node.state):   # check goal state
            return node

        for state, _ in operators_func(node.state):   # go through next states
            # create tree node with the new state
            child_node = TreeNode(state)


            # link child node to its parent in the tree
            node.add_child(child_node)


            # enqueue the child node
            queue.append(child_node)
            


    return None

We can now use this function to actually perform a breadth-first search on the buckets problem: we pass it the initial state, our goal condition function, and the function for obtaining child states.

In [29]:
goal = breadth_first_search(BucketState(0,0),
                            goal_bucket_state,
                            child_bucket_states)
print(goal.state)

(2, 3)


In order to print the actual steps from the initial state to the last, we can take advantage of each node's link to its parent.

In [30]:
def print_solution(node:TreeNode):
    if node:
        if node.parent:
            print_solution(node.parent)
        print(node.state)
    else:
        print("no solution found")

    return

Now we can print the solution:

In [31]:
print_solution(goal)

(0, 0)
(4, 0)
(1, 3)
(1, 0)
(0, 1)
(4, 1)
(2, 3)


If we need a description for each of the employed operators, we could have each operation function return also such a description, and modify the TreeNode class so that each node also includes a description of the edge to get there. We leave that as an exercise after class.

##### Depth-first search

Implement depth-first search (again, in a manner that is independent of the problem at hand). You can start from your breadth-first search implementation and with minor changes get an implementation for depth-first search.

In [32]:
def depth_first_search_logic(current_node:TreeNode, goal_state_func, operators_func, visited:set):
    visited.add(current_node.state)

    if goal_state_func(current_node.state):
        return current_node
    
    for state, _ in operators_func(current_node.state):
        child_node = TreeNode(state)
        if state not in visited:
            if depth_first_search_logic(child_node, goal_state_func, operators_func, visited):
                current_node.add_child(child_node)
                return current_node

    return None

def depth_first_search(initial_state, goal_state_func, operators_func):
    cur_node = depth_first_search_logic(TreeNode(initial_state), goal_state_func, operators_func, set())
    while (len(cur_node.children) > 0):
        cur_node = cur_node.children[0]
    return cur_node
    

Test it on the two buckets problem.

In [33]:
goal = depth_first_search(BucketState(0,0),
                            goal_bucket_state,
                            child_bucket_states)
print_solution(goal)


(0, 0)
(4, 0)
(4, 3)
(0, 3)
(3, 0)
(3, 3)
(4, 2)
(0, 2)
(2, 0)


If you are unable to get a solution, think about it: depth-first search is not a complete search method, and one of the reasons for that is if the state space contains cycles. As such, you need to make sure you avoid entering into a cycle by keeping a visited nodes list or set and checking that list whenever you generate a new state.

##### Depth-limited search

Another way to make it work is to impose a depth limit to the problem. Implement depth-limited search.

In [34]:
def depth_limited_search_logic(current_node:TreeNode, goal_state_func, operators_func, depth_limit, visited:set):
    visited.add(current_node.state)

    if depth_limit < 1:
        return None
    if goal_state_func(current_node.state):
        return current_node
    
    for state, _ in operators_func(current_node.state):
        child_node = TreeNode(state)
        if state not in visited:
            if depth_limited_search_logic(child_node, goal_state_func, operators_func, depth_limit-1,visited):
                current_node.add_child(child_node)
                return current_node

    return None

def depth_limited_search(initial_state, goal_state_func, operators_func, depth_limit):
    cur_node = depth_limited_search_logic(TreeNode(initial_state), goal_state_func, operators_func,depth_limit, set())
    if cur_node:
        while (len(cur_node.children) > 0):
            cur_node = cur_node.children[0]
    return cur_node

Test it on the two buckets problem.

In [35]:
goal = depth_limited_search(BucketState(0,0),
                            goal_bucket_state,
                            child_bucket_states,6)
print_solution(goal)


no solution found


##### Iterative deepening search

Based on depth-limited, you can easily implement iterative-deepening search.

In [36]:
def iterative_deepening_search(initial_state, goal_state_func, operators_func, depth_limit):
    for l in range(depth_limit+1):
        node = depth_limited_search(initial_state,goal_state_func, operators_func,l)
        if node:
            return node, l
    return None, _





Again, test it on the two buckets problem.

In [41]:
goal, depth = iterative_deepening_search(BucketState(0,0),
                            goal_bucket_state,
                            child_bucket_states,10)
print_solution(goal)
if goal:
    print(depth)


(0, 0)
(4, 0)
(1, 3)
(1, 0)
(0, 1)
(4, 1)
(2, 3)
7


## The Missionaries and Cannibals Problem

<img src="https://allfish24.files.wordpress.com/2016/09/gambar-1.jpg">

Three missionaries and three cannibals are on one of the banks of the river with a boat that only takes one or two people. The boat cannot travel the river alone.

The goal is to find a way to get the six to the other bank of the river without ever having more cannibals than missionaries on one of the banks (even at the instant they leave/join the boat) during the process.

**a)** Formulate this problem as a search problem by defining the state representation, initial state, operators (their name, preconditions, effects, and cost), and final state(s).

**b)** Draw a partial state space graph with some of the possible transitions. What is the state space size for this problem? Consider only reachable states. *(Note: you might want to do this on paper and pencil.)*

**c)** Solve the problem by hand, using a search tree. What solutions have you found?

**d)** Make your own implementation of the problem as a search problem and take advantage of the implemented search algorithms to find solutions.

In [ ]:
class MisCanState:

    def __init__(self, mis, can, boat):
        self.mis = mis   # number of missionaries on the initial bank
        self.can = can   # number of canibals on the initial bank
        self.boat = boat   # boat on the initial bank?

    '''needed for the visited list'''
    def __eq__(self, other):
        if isinstance(other, self.__class__):
            return self.__dict__ == other.__dict__
        else:
            return False

    def __ne__(self, other):
        """Overrides the default implementation (unnecessary in Python 3)"""
        return not self.__eq__(other)

    def __hash__(self):
        return hash((self.mis, self.can, self.boat))
    ''' - '''

    def __str__(self):
        return "(" + str(self.mis) + ", " + str(self.can) + ", " + str(self.boat) + ")"

In [39]:
#loadM1          S(m1,c1,_,_,mb,cb,b) b = 1, mb + cb < boatCapacity, (m1 > c1 | m1 = 1), m1 > 0  S(m1-1,c1,_,_,mb+1,cb,b)
#loadM2          S(_,_,m2,c2,mb,cb,b) b = 2, mb + cb < boatCapacity, (m2 > c2 | m2 = 1), m2 > 0  S(_,_,m2-1,c2,mb+1,cb,b)
#loadC1          S(_,c1,_,_,mb,cb,b) b = 1, mb + cb < boatCapacity, c1 > 0                       S(_,c1-1,_,_,mb,cb+1,b)
#loadC2          S(_,_,_,c2,mb,cb,b) b = 2, mb + cb < boatCapacity, c2 > 0                       S(_,_,_,c2-1,mb,cb+1,b)
#unloadM1        S(m1,c1,_,_,mb,_,b) b = 1, m1 + 1 >= c1, mb > 0                                 S(m1+1,c1,_,_,mb-1,_,b)
#unloadM2        S(_,_,m2,c2,mb,_,b) b = 2, m2 + 1 >= c2, mb > 0                                 S(_,_,m2+1,c2,mb-1,_,b)
#unloadC1        S(m1,c1,_,_,_,cb,b) b = 1, cb > 0, (c1 + 1 <= m1 | m1 = 0)                      S(m1,c1+1,_,_,_,cb+1,b)
#unloadC2        S(_,_,m2,c2,_,cb,b) b = 2, cb > 0, (c2 + 1 <= m2 | m2 = 0)                      S(_,_,m2,c2+1,_,cb+1,b)
#moveBoat12      S(_,_,_,_,_,_,b) b = 1                                                          S(_,_,_,_,_,_,2)
#moveBoat21      S(_,_,_,_,_,_,b) b = 2                                                          S(_,_,_,_,_,_,1)

def loadM1(state:MisCanState):    
    


_IncompleteInputError: incomplete input (1237037951.py, line 13)